# <font color='black'> Регрессионный анализ: продолжение, 2026 </font>
## <font color='black'> Практическое занятие 3. Синтетический контроль </font>

Источник данных: Alberto Abadie, Alexis Diamond & Jens Hainmueller (2010) Synthetic Control Methods for Comparative Case Studies: Estimating the Effect of California’s Tobacco Control Program, Journal of the American Statistical Association, 105:490, 493-505

Ссылка на статью: https://disk.yandex.ru/i/0ALu3UjyDrNTQw

In [3]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import plotly.express as px
from scipy.optimize import minimize

In [4]:
df = pd.read_stata('https://github.com/microsoft/SparseSC/raw/master/replication/smoking.dta')
df = df.drop(columns=["lnincome", "beer", "age15to24"])
df.head()

,state,year,cigsale,retprice
0,Rhode Island,1970.0,123.900002,39.299999
1,Tennessee,1970.0,99.800003,39.900002
2,Indiana,1970.0,134.600006,30.600000
3,Nevada,1970.0,189.500000,38.900002
4,Louisiana,1970.0,115.900002,34.299999


Предварительно оценим модель DiD, уже знакомую Вам. Сгенерим дамми-переменную, которая отвечает за группу воздействия, и дамми-переменную для периода (1 - после введения антитабачной программы в Калифорнии в 1988, 0 - соответственно, до).    

In [5]:
df['treated'] = df['state'].apply(lambda x: 1 if x == "California" else 0)
df['after'] = df['year'].apply(lambda x: 1 if x > 1988 else 0)

Проинтерпретируйте полученные оценки в модели DiD:

In [6]:
did_model = smf.ols("cigsale ~ after*treated", data=df).fit(cov_type = "HC3")
print(did_model.summary())

                            OLS Regression Results                            
Dep. Variable:                cigsale   R-squared:                       0.207
Model:                            OLS   Adj. R-squared:                  0.205
Method:                 Least Squares   F-statistic:                     172.0
Date:                Sun, 11 Oct 2026   Prob (F-statistic):           8.16e-93
Time:                        20:44:25   Log-Likelihood:                -5793.3
No. Observations:                1209   AIC:                         1.159e+04
Df Residuals:                    1205   BIC:                         1.161e+04
Df Model:                           3                                         
Covariance Type:                  HC3                                         
                    coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------
Intercept       130.5695      1.218    107.224

Для последующего построения графика преобразуем вид данных таким образом, чтобы каждая строка соответствовала штату, столбец - году.

In [7]:
df_t = df.pivot(index= 'state', columns = 'year', values = "cigsale")
df_t.head()

year,1970.0,1971.0,1972.0,1973.0,1974.0,1975.0,1976.0,1977.0,1978.0,1979.0,...,1991.0,1992.0,1993.0,1994.0,1995.0,1996.0,1997.0,1998.0,1999.0,2000.0
state,,,,,,,,,,,,,,,,,,,,,
Alabama,89.800003,95.400002,101.099998,102.900002,108.199997,111.699997,116.199997,117.099998,123.000000,121.400002,...,107.900002,109.099998,108.500000,107.099998,102.599998,101.400002,104.900002,106.199997,100.699997,96.199997
Arkansas,100.300003,104.099998,103.900002,108.000000,109.699997,114.800003,119.099998,122.599998,127.300003,126.500000,...,116.800003,126.000000,113.800003,108.800003,113.000000,110.699997,108.699997,109.500000,104.800003,99.400002
California,123.000000,121.000000,123.500000,124.400002,126.699997,127.099998,128.000000,126.400002,126.099998,121.900002,...,68.699997,67.500000,63.400002,58.599998,56.400002,54.500000,53.799999,52.299999,47.200001,41.599998
Colorado,124.800003,125.500000,134.300003,137.899994,132.800003,131.000000,134.199997,132.000000,129.199997,131.500000,...,90.199997,88.300003,88.599998,89.099998,85.400002,83.099998,81.300003,81.199997,79.599998,73.000000
Connecticut,120.000000,117.599998,110.800003,109.300003,112.400002,110.199997,113.400002,117.300003,117.500000,117.400002,...,86.699997,83.500000,79.099998,76.599998,79.300003,76.000000,75.900002,75.500000,73.400002,71.400002


Построим график динамики показателя продаж сигарет в Калифорнии (группа воздействия) и остальных штатах, составляющих контрольную группу. Для контрольной группы усредним значения показателя продаж сигарет по временным периодам. Какие предварительные выводы Вы можете сделать по этому графику?  

In [8]:
plot_df = df_t.loc[df_t.index == "California"].T.reset_index(drop=False)
plot_df["Untreated"] = df_t.loc[df_t.index != "California"].mean(axis=0).values

plot_df = plot_df.rename(columns={"California": "Калифорния", "Untreated": "Остальные штаты"})

fig = px.line(
    plot_df,
    x="year",
    y=["Калифорния", "Остальные штаты"],
    labels={"year": "Год", "value": "Продажи сигарет", "variable": ""},
    color_discrete_map={"Калифорния": "red", "Остальные штаты": "blue"},
)

fig.add_vline(x=1988, line_dash="dash", line_color="grey")

fig.update_layout(title="Динамика продаж сигарет на душу населения")

fig.show()

Реализуем метод синтетического контроля и посчитаем ATT (average treatment effect on the treated)

In [16]:
# Шаг 1 предполагает подготовку данных к реализации метода синтетического контроля

before = df_t.loc[:, df_t.columns <= 1988].values # данные о продажах сигарет до введения антитабачной программы 1988 г. Мы их используем для получения значений
                                                     # синтетического контроля
after  = df_t.loc[:, df_t.columns > 1988].values # соответственно, данные о продажах сигарет после введения антитабачной программы 1988 г., с которыми мы будем сравнивать
                                                     # полученные значения синтетического контроля для оценки среднего эффекта воздействия (ATT)

# Калифорния — группа воздействия, остальные штаты — объекты из контрольной группы
i = df_t.index.get_loc("California")  # получаем индекс строки Калифорнии в массиве данных
m = np.arange(len(df_t)) != i # True — все штаты кроме Калифорнии
X = before[m].T # задаем в матрице значения признака продаж сигарет для контрольных штатов

# Шаг 2: находим оптимальные веса для получения синтетического контроля
def fit_sc(X, y):
    r = minimize(lambda w: np.mean((X @ w - y) ** 2), # зададим функцию потерь через RMSE (отклонения группы воздействия от взвешенной комбинации признаков контрольных штатов)
                 np.ones(X.shape[1]) / X.shape[1], method="SLSQP",  # стартовые значения - равные веса, а далее уже минимизируем RMSE;
                 # используем в качестве метода оптимизации Sequential Least Squares Quadratic Programming, так как данный метод учитывает введенные ограничения
                 bounds=[(0, 1)] * X.shape[1],
                 constraints={"type": "eq", "fun": lambda w: w.sum() - 1}) # указывает тип ограничения - равенство (equality constraint),
                                                     # то есть, функция ограничения равна 0;
                                                     # ссылаемся через 'fun': constraint на функцию np.sum(w) - 1, заданную ранее
    return r.x

w = fit_sc(X, before[i])

# Шаг 3: считаем ATT
treatment_effect = after[i] - after[m].T @ w
ATT = treatment_effect.mean()
print(f"ATT: {ATT:.2f}")

ATT: -19.51


Убедимся в том, что полученный эффект не является случайным. Для этого реализуем плацебо-тест

In [18]:
placebo = [np.mean(after[j] - after[m].T @ fit_sc(X, before[j])) for j in np.where(m)[0]]
p = (np.abs(placebo) >= abs(ATT)).mean() # Какая доля случайных "эффектов" плацебо по величине превышает соответствующий эффект для Калифорнии?

print(f"ATT: {ATT:.2f}, p-value: {p:.4f}") # Если бы p-value превышал конвенциональный уровень значимости,
                                           # то можно было бы предполагать случайный эффект, полученный для Калифорнии

ATT: -19.51, p-value: 0.0000
